  # PostProcessing
  This notebook is initially designed for PostProcessing ASETS-II numerical and experimental data.
  It can also be used for other configurations's numerical results. But if you want to compare with other experimental data with a different format than ASETS-II. You should find a way to read them to RTD and RTDt in this notebook.

In [ ]:
using ComputationalHeatTransfer
using Plots
using Interact

  # Read simulation data

In [ ]:
using JLD2
using FileIO

read_path = "D:/CURRENT/solution_diy_24_turns_0.01_fill_50W_EH.jld2"
SimuResult = load(read_path)["simulationResult"];

  ### get time array

In [ ]:
t = SimuResult.tube_hist_t;

  # Plot 2D graphs

  ### film and slug dynamics

In [ ]:
# 1. Define bounds to fit the whole 0.635m x 0.4064m pipe with a small margin
x_zoom = (-0.35, 0.35) 
y_zoom = (-0.15, 0.55) 

# 2. Generate the full-system GIF 
@gif for i in 1:10:length(t) # Skipping every 5 frames so the 7s run doesn't take forever to render
    plot(OHPSlug(), i, SimuResult, 
         size=(840, 600),         # Canvas sized to match the physical width/height ratio
         aspect_ratio=1,          # Locks the geometry so the pipe doesn't look stretched
         xlims=x_zoom,            
         ylims=y_zoom,            
         margin=5Plots.mm)        
end

  ### plate T [K]

In [ ]:
@gif for i in 1:20:length(t)
    plot(OHPTemp(), i, SimuResult, clim=(291.2, 292.6))
end

In [ ]:
@gif for i in 1:10:length(t)
    plot(OHPTemp(), i, SimuResult, clim=(291.2, 293))
end

  ### 2D superheat

In [ ]:
@gif for i in eachindex(t)
    plot(OHPSuper(),i,SimuResult)
end

  ### 2D pressure

In [ ]:
@gif for i in eachindex(t)
    plot(OHPPres(),i,SimuResult)
end

  # Plot 2D interpolated curves

  ### Interpolate 2D T data from the plate for fixed sensors on the plate

  place the 2D sensors

In [ ]:
x2Dsensors = [-0.25, 0.0, 0.25] 
y2Dsensors = [0.2, 0.2, 0.2]
plate_sensors = (x2Dsensors,y2Dsensors);

  get the curve

In [ ]:
t_hist,g_hist = getTcurve(plate_sensors,SimuResult);

  ### Read experiment T data
  This part can be customized as long as you can get a matrix for RTD (sensor data) and an array for RTDt (time points)

In [ ]:
import XLSX

 read experiment file for ASETS-II data.

In [ ]:
expfile = expfileDict["O002_H001_P040"]
exppath = "../expdata/"
xf = XLSX.readxlsx(exppath*expfile);

  get experiment data for ASETS-II data

In [ ]:
Onum, Hnum, power_exp = getconfig(expfile)
RTDt,RTD = getRTD(xf,Onum);

  ### 2D interpolated temperature curve at fixed sensors

In [ ]:
RTD_for_plotting = [1, 2, 3]

plot OHP

In [ ]:
plot(OHP(),SimuResult)

plot sensors

In [ ]:
# Overlay the scatter points for the sensors
scatter!(x2Dsensors[RTD_for_plotting], y2Dsensors[RTD_for_plotting], label="Sensors", color=:red)

# Annotate the sensors with their numbers, shifting the y-position slightly so text doesn't overlap the dot
annotate!(x2Dsensors[RTD_for_plotting], y2Dsensors[RTD_for_plotting] .- 0.015, RTD_for_plotting)

plot temperature curve

In [ ]:
plot(OHPcurve(), RTD_for_plotting, (t_hist, g_hist), SimuResult)
plot!(OHPTexp() ,RTD_for_plotting,(RTDt,RTD)     ,SimuResult)

  ### 2D interpolated thermal conductance

In [ ]:
ihot = 4 # hot sensor  for calculating thermal conductance
icold = 8 # cold sensor  for calculating thermal conductance;

plot them separately

In [ ]:
plot(OHPCond(),(ihot,icold),(t_hist,g_hist),(RTDt,RTD),SimuResult)

  ### Liquid slug velocity statistics

fix title and ylabel and legend

In [ ]:
plot(OHPV(), SimuResult::SimulationResult,ylimit=(-2,2))

  # Plot 1D interpolated curves

In [ ]:
@manipulate for i in 1:1:length(t)
    plot(OHP1DT(),i,SimuResult,xlim=(1,2))
    plot!(twinx(),OHPTwall(),i,SimuResult,xlim=(1,2))
#     plot!(twinx(),OHP1DΔT(),i,SimuResult,xlim=(1,2))
#     plot!(twinx(),OHP1DP(),i,SimuResult,xlim=(1,2))
end

  ### 1D sensor selector

In [ ]:
L = SimuResult.integrator_tube.p.tube.L
@manipulate for ξ in 0:1e-3:L
    plot(OHP(),SimuResult) # plot the ohp layout

    xprobe,yprobe = oneDtwoDtransform(ξ,SimuResult)
    scatter!([xprobe],[yprobe])
end

  # Plot 1D property curve for a fixed location sensor

In [ ]:
xsensors1D = [2.097, 3.0, 4,4.1]

θhist1D,phist1D = get1DTandP(xsensors1D, SimuResult);

plot(t,θhist1D,label=string.("ξ=", xsensors1D'),xlabel="time [s]", ylabel="temperature [K]")

  ### get boiling data (if there are any)

In [ ]:
if length(SimuResult.boil_hist) != 0
boil_data,boil_num_x,boil_num_t,t_boil,x2D_boil,y2D_boil,boil_dt = get_boil_matrix(SimuResult::SimulationResult);
end

boiling frequency scatter graph (if there are any)

In [ ]:
plt = plot()
if length(SimuResult.boil_hist) != 0
plt = plot(OHP(),SimuResult)
scatter!(x2D_boil,y2D_boil,
    colorbar=true,markeralpha=delta.(boil_num_x),colorbar_title="\n boiling frequency [Hz]",right_margin=3Plots.mm,marker_z=boil_num_x./SimuResult.tube_hist_t[end],markerstrokewidth=0,markercolor=cgrad(:greys, rev = true))
end
plt

boiling frequency curve (if there are any)

In [ ]:
plt = plot()
if length(SimuResult.boil_hist) != 0
plt = plot(t_boil,boil_num_t./boil_dt,
color=:orange, legend=:topleft, ylabel="f [HZ]",xlabel="time [s]", label="overall boiling frequency")
end
plt

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*

In [ ]:
using JLD2
using FileIO
using Statistics
using DataFrames
using Plots 
using ComputationalHeatTransfer # Assumes getTcurve is here

# 1. Define Physical Parameters
plate_d = 6.35e-3              
width_ohp = 11.875 * 0.0254    
height_ohp = 20.0 * 0.0254     
power = 50.0                   
directional_power = power      
cross_section_A = width_ohp * plate_d

# Centroids (Assuming origin at bottom-left)
hot_x = 9.875 * 0.0254; hot_y = 2.0 * 0.0254
cold_x = (11.875 / 2.0) * 0.0254; cold_y = (20.0 / 2.0) * 0.0254
distance_L = sqrt((hot_x - cold_x)^2 + (hot_y - cold_y)^2) # ~0.226 m

plate_sensors = ([hot_x, cold_x], [hot_y, cold_y])
ihot = 1; icold = 2 

# 2. File List
file_list = [
]

println("Starting plotting sequence...")

# 3. Loop and Plot
for file in file_list
    if !isfile(file)
        println("  ⚠️ Skipping (File not found): ", basename(file))
        continue
    end
    
    # Extract turns for the title
    turn_match = match(r"_(\d+)_turns", basename(file))
    turns = turn_match !== nothing ? parse(Int, turn_match.captures[1]) : 0

    try
        # Load and extract data
        SimuResult = load(file)["simulationResult"]
        t_hist, g_hist = getTcurve(plate_sensors, SimuResult)
        
        T_hot = g_hist[:, ihot]
        T_cold = g_hist[:, icold]
        delta_T = T_hot .- T_cold
        
        # Define steady-state region (last 30%)
        total_points = length(t_hist)
        ss_start = round(Int, 0.70 * total_points)
        time_ss_start = t_hist[ss_start]
        
        # --- THE CORRECTED MATH ---
        # Average the temperatures during steady state FIRST
        mean_delta_T_ss = mean(delta_T[ss_start:end])
        correct_k_eff = (directional_power * distance_L) / (cross_section_A * mean_delta_T_ss)
        
        # --- PLOTTING ---
        # Plot 1: Raw Temperatures
        p1 = plot(t_hist, T_hot, label="Heater Temp", ylabel="Temp (K)", title="$(turns) Turns: Thermal Profile", lw=2, color=:firebrick)
        plot!(p1, t_hist, T_cold, label="Condenser Temp", lw=2, color=:steelblue)
        vline!(p1, [time_ss_start], label="Steady State Start", ls=:dash, color=:black)

        # Plot 2: Delta T
        p2 = plot(t_hist, delta_T, label="ΔT", ylabel="ΔT (K)", color=:purple, lw=1.5)
        hline!(p2, [0], label="Zero ΔT (Singularity Risk)", color=:red, ls=:dot, lw=2)
        vline!(p2, [time_ss_start], label="", ls=:dash, color=:black)

        # Plot 3: The K_eff Comparison
        # Calculate instantaneous k_eff (the flawed method) just to visualize the spikes
        instant_k = (directional_power .* distance_L) ./ (cross_section_A .* replace(delta_T, 0.0 => 1e-5))
        
        p3 = plot(t_hist, instant_k, label="Instantaneous k_eff (Flawed)", ylabel="k_eff (W/mK)", xlabel="Time (s)", color=:orange, alpha=0.6)
        # Cap the Y-axis so the graph is readable, even if it spikes to infinity
        ylims!(p3, (0, max(correct_k_eff * 4, 20000))) 
        hline!(p3, [correct_k_eff], label="Time-Averaged k_eff = $(round(correct_k_eff, digits=1))", color=:green, lw=2)
        vline!(p3, [time_ss_start], label="", ls=:dash, color=:black)

        # Assemble and save
        combined_plot = plot(p1, p2, p3, layout=(3,1), size=(800, 900), margin=5Plots.mm)
        filename = "OHP_Diagnostic_$(turns)_Turns.png"
        savefig(combined_plot, filename)
        
        println("  ✅ Generated Plot: ", filename, " (k_eff = ", round(correct_k_eff, digits=2), ")")
        
    catch e
        println("  ❌ Error processing ", basename(file), ": ", e)
    end
end

println("All plots generated successfully.")

In [ ]:
using JLD2
using FileIO
using Statistics
using DataFrames
using Plots 

# 1. Define Physical Parameters 
plate_d = 6.35e-3              
width_ohp = 0.301625           # 11.875 inches
length_ohp = 0.508             # 20 inches total
power = 50.0                   
directional_power = power      
cross_section_A = width_ohp * plate_d

# Centroids for distance calculation
hot_center_x = 0.0254          # Center of 2-inch heater
cold_center_x = 0.2794         # Center of 18-inch condenser
distance_L = cold_center_x - hot_center_x  # exactly 10 inches (0.254 m)

# --- DATA TABLE SETUP ---
results_df = DataFrame(
    Fill_Ratio = Float64[],
    Q_W = Float64[],
    d_m = Float64[],
    A_m2 = Float64[],
    Delta_T_ss = Float64[],
    K_eff = Float64[]
)

# 2. File List
file_list = [
]

println("Starting plotting sequence...")

# 3. Loop and Plot
for file in file_list
    if !isfile(file)
        println("  ⚠️ Skipping (File not found): ", basename(file))
        continue
    end
    
    turn_match = match(r"_(\d+)_turns", basename(file))
    turns = turn_match !== nothing ? parse(Int, turn_match.captures[1]) : 0

    fill_match = match(r"_([0-9\.]+)_fill", basename(file))
    fill_ratio = fill_match !== nothing ? parse(Float64, fill_match.captures[1]) : NaN

    try
        # Load and extract direct fields
        SimuResult = load(file)["simulationResult"]
        
        t_hist = SimuResult.tube_hist_t
        T_plate = SimuResult.plate_T_hist 
        
        # --- 2D SPATIAL MASKING ---
        num_timesteps = length(t_hist)
        
        # Extract the first frame as a standard Matrix to get physical dimensions
        # The broadcasting Float64.() bypasses the 'MethodError' by forcing 
        # the Primal nodes struct to dump its raw numerical values.
        first_frame = Float64.(T_plate[1])
        Nx, Ny = size(first_frame)
        
        # Calculate how many X-nodes fall into the 2-inch heater zone
        # 0.0508 m / 0.508 m = 10% of the plate length
        hot_cutoff_idx = round(Int, (0.0508 / length_ohp) * Nx) 
        
        # Initialize arrays for the time-history of the averages
        T_hot_spatial_avg = zeros(num_timesteps)
        T_cold_spatial_avg = zeros(num_timesteps)
        
        # Loop through time and calculate the spatial mean for each zone
        for i in 1:num_timesteps
            # Convert the current timestep's grid to a standard 2D Float64 Matrix
            current_frame = Float64.(T_plate[i])
            
            # Slice the matrix: Rows 1 to cutoff for heater, the rest for condenser
            T_heater_zone = current_frame[1:hot_cutoff_idx, :]
            T_condenser_zone = current_frame[hot_cutoff_idx+1:end, :]
            
            T_hot_spatial_avg[i] = mean(T_heater_zone)
            T_cold_spatial_avg[i] = mean(T_condenser_zone)
        end
        
        # Calculate bulk Delta T over time
        delta_T = T_hot_spatial_avg .- T_cold_spatial_avg
        
        # Define steady-state region (last 30%)
        ss_start = round(Int, 0.70 * num_timesteps)
        time_ss_start = t_hist[ss_start]
        
        # Temporal Average: Mean of the spatial Delta T during steady state
        mean_delta_T_ss = mean(delta_T[ss_start:end])
        correct_k_eff = (directional_power * distance_L) / (cross_section_A * mean_delta_T_ss)
        
        # Push to Debugging DataFrame
        push!(results_df, (fill_ratio, directional_power, distance_L, cross_section_A, mean_delta_T_ss, correct_k_eff))
        
        # --- PLOTTING ---
        p1 = plot(t_hist, T_hot_spatial_avg, label="Avg Heater Temp", ylabel="Temp (K)", title="$(turns) Turns, $(fill_ratio) Fill: Thermal Profile", lw=2, color=:firebrick)
        plot!(p1, t_hist, T_cold_spatial_avg, label="Avg Condenser Temp", lw=2, color=:steelblue)
        vline!(p1, [time_ss_start], label="Steady State Start", ls=:dash, color=:black)

        p2 = plot(t_hist, delta_T, label="Bulk ΔT", ylabel="ΔT (K)", color=:purple, lw=1.5)
        hline!(p2, [0], label="Zero ΔT", color=:red, ls=:dot, lw=2)
        vline!(p2, [time_ss_start], label="", ls=:dash, color=:black)

        instant_k = (directional_power .* distance_L) ./ (cross_section_A .* replace(delta_T, 0.0 => 1e-5))
        
        p3 = plot(t_hist, instant_k, label="Instantaneous k_eff", ylabel="k_eff (W/mK)", xlabel="Time (s)", color=:orange, alpha=0.6)
        ylims!(p3, (0, max(correct_k_eff * 4, 20000))) 
        hline!(p3, [correct_k_eff], label="Time-Averaged k_eff = $(round(correct_k_eff, digits=1))", color=:green, lw=2)
        vline!(p3, [time_ss_start], label="", ls=:dash, color=:black)

        combined_plot = plot(p1, p2, p3, layout=(3,1), size=(800, 900), margin=5Plots.mm)
        filename = "OHP_Diagnostic_$(turns)_Turns_$(fill_ratio)_Fill.png"
        savefig(combined_plot, filename)
        
        println("  ✅ Generated Plot: ", filename)
        
    catch e
        println("  ❌ Error processing ", basename(file), ": ", e)
    end
end

println("\nAll plots generated successfully.")

# --- DISPLAY THE TABLE ---
println("\n================ K_eff Diagnostic Table ================")
sort!(results_df, :Fill_Ratio)
display(results_df)
println("========================================================")

In [ ]:
using JLD2
using Plots 
using Statistics
using ComputationalHeatTransfer # Assumes getTcurve is here

# --- Output Directory Setup ---
out_dir = "D:/CURRENTRESULTD"
mkpath(out_dir) 

# 1. Define Physical Bounds
width_ohp = 0.3048             
length_ohp = 0.5588            

# 2. Manual File and Fill Ratio Lists
# Make sure these two lists align perfectly!
file_list = [
]

# Provide the fill ratios manually as strings or numbers
fill_ratios = [
]

# 3. Create a dense grid of virtual sensors
nx = 100 
ny = 60  

xs = range(0, stop=length_ohp, length=nx)
ys = range(0, stop=width_ohp, length=ny)

sensor_xs = Float64[]
sensor_ys = Float64[]

for x in xs
    for y in ys
        push!(sensor_xs, x)
        push!(sensor_ys, y)
    end
end

sensor_tuple = (sensor_xs, sensor_ys)

println("Starting 1% Time-Averaged heatmap batch processing...")
println("➔ Setup complete. Reaching the loop...")
flush(stdout) # Forces the terminal to display this immediately

# 4. Loop and Plot using zip()
for (file, fill_ratio) in zip(file_list, fill_ratios)
    
    println("--------------------------------------------------")
    println("➔ INSIDE LOOP: Targeting file with fill ratio: ", fill_ratio)
    flush(stdout)

    if !isfile(file)
        println("  ⚠️ Skipping (File not found): ", basename(file))
        flush(stdout)
        continue
    end
    
    base_name = basename(file)

    try
        println("  ⏳ Attempting to load .jld2 file into memory (This might take a while)...")
        flush(stdout)
        
        # Load data
        SimuResult = load(file)["simulationResult"]
        
        println("  ✅ File loaded! Inspecting internal data structure...")
        println("  -> Fields inside SimuResult: ", fieldnames(typeof(SimuResult)))
        flush(stdout)
        
        println("  ⏳ Attempting to extract getTcurve data (Warning: May freeze here)...")
        flush(stdout)
        
        # Extract temperature histories
        t_hist, g_hist = getTcurve(sensor_tuple, SimuResult)
        
        println("  ✅ Data extracted! Calculating time-averages...")
        flush(stdout)
        
        # Calculate the last 1% window
        total_steps = length(t_hist)
        start_idx = round(Int, 0.99 * total_steps)
        g_last_1_percent = g_hist[start_idx:end, :]
        T_mean_1D = vec(mean(g_last_1_percent, dims=1))
        
        println("  ✅ Averages calculated! Building 2D array...")
        flush(stdout)
        
        # Reshape the 1D time-averaged data back into a 2D spatial matrix
        T_2D = zeros(ny, nx)
        idx = 1
        for i in 1:nx
            for j in 1:ny
                T_2D[j, i] = T_mean_1D[idx]
                idx += 1
            end
        end
        
        println("  ✅ Array built! Generating plot (Julia might compile here)...")
        flush(stdout)
        
        # Generate the Heatmap
        p = contourf(xs, ys, T_2D, 
            levels=30,                   
            c=:inferno,                  
            xlabel="Plate Length (m)", 
            ylabel="Plate Width (m)", 
            title="$(fill_ratio) Fill Ratio: Avg Temp (Last 1% of Sim)",
            aspect_ratio=:equal,         
            right_margin=15Plots.mm,     
            colorbar_title="Time-Averaged Temperature (K)",
            lw=0                         
        )

        filename = "OHP_TimeAvg_Heatmap_$(fill_ratio)_Fill.png"
        full_save_path = joinpath(out_dir, filename)
        
        savefig(p, full_save_path)
        println("  ✅ SUCCESS: Saved Heatmap: ", full_save_path)
        flush(stdout)
        
    catch e
        println("  ❌ Error processing ", base_name, ": ", e)
        flush(stdout)
    end
end

println("All time-averaged heatmaps generated successfully.")
flush(stdout)